## The DAG — tasks & dependencies

Tasks declare which other tasks must succeed first. Lakeflow Jobs builds a **directed acyclic graph** from those dependencies, runs tasks in topological order, and parallelises wherever it can.

The bank's nightly job:

```text
              ┌── ingest_cards ───┐
start_marker ─┼── ingest_accounts ┼─► silver_build ─► gold_customer_360 ─► refresh_dashboard ─► notify
              └── ingest_loans ───┘
```

- The three `ingest_*` tasks run **in parallel**.
- `silver_build` lists all three as upstream — it runs once all three succeed.
- `refresh_dashboard` runs after `gold_customer_360`.
- `notify` runs **unconditionally** on completion (see *Run if* below).

**Three task settings the exam tests:**

- **`depends_on`** — the list of upstream task keys. An empty list makes a **root** task.
- **Run if** — when a task should run *given* upstream outcomes: `ALL_SUCCESS` (default), `AT_LEAST_ONE_SUCCESS`, `NONE_FAILED`, `ALL_DONE`, `AT_LEAST_ONE_FAILED`, `ALL_FAILED`. **`ALL_DONE` is how you wire a final notification task that always fires**, success or failure.
- **`max_retries`** — how many times to re-attempt the task on failure, with exponential backoff between attempts.

The DAG is what makes a job *self-parallelising*: you declare dependencies, not order, and the scheduler figures out what can run at the same time.
